# Visualize robot-controller training

This notebook reads recipe status and metrics without loading model weights. Re-run the final cell to refresh an active run.

In [ ]:
import json
from pathlib import Path
import sys
sys.path.append("..")
from IPython.display import display
import matplotlib.pyplot as plt
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from robot_controller.visualize_trec import TrainingRecipeVisualizer

In [ ]:
creation_style = "exist-ok"
expruns_path = None
results_path = None
experiment_name = "robot_controller_training"
run_name = "trec_vae_neo_lstm_mdn_sample"

if expruns_path:
    expruns_path = Path(expruns_path)
    if not expruns_path.exists():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    for name in [experiment_name, "robot_controller", "sensorprocessing_conv_vae_neo", "sensorprocessing_propriotuned_cnn"]:
        Config().copy_experiment(name)
if results_path:
    results_path = Path(results_path)
    if not results_path.exists():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment_name, run_name, creation_style=creation_style)

In [ ]:
display(TrainingRecipeVisualizer(exp).build())
status_path = exp.data_dir() / "recipe_status.json"
if status_path.is_file():
    print(json.dumps(json.loads(status_path.read_text()), indent=2))
metrics_path = exp.data_dir() / "metrics.jsonl"
if metrics_path.is_file():
    records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line]
    for stage in dict.fromkeys(record["stage"] for record in records):
        selected = [record for record in records if record["stage"] == stage]
        validation_key = "validation_nll" if "validation_nll" in selected[0] else "validation_loss"
        plt.plot([record["epoch"] for record in selected], [record[validation_key] for record in selected], marker=".", label=stage)
    plt.xlabel("Stage epoch")
    plt.ylabel("Validation objective")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()